In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
from matplotlib.lines import Line2D


In [ ]:
# Paths
base_path  = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
output_dir = base_path / "Outputs"



# GLOBAL TROPICAL RESTORATION COSTS FROM BODIN ET AL

In [ ]:
# ---- 1) Unit costs (USD/ha) ----
costs = pd.DataFrame({
    "restoration_type": ["ANR", "Planted"],
    "cost_min_usd_per_ha": [12, 105],
    "cost_max_usd_per_ha": [3880, 25830],
})
costs["cost_mid_usd_per_ha"] = costs[["cost_min_usd_per_ha", "cost_max_usd_per_ha"]].mean(axis=1)
costs

In [ ]:
# Load: prefer Parquet, otherwise CSV
try:
    areas = pd.read_parquet(output_dir / "forest_area_totals.parquet")
except Exception:
    areas = pd.read_csv(output_dir / "forest_area_totals.csv")

def ha(metric: str) -> float:
    return float(areas.loc[areas["metric"].eq(metric), "area_m2"].iloc[0]) / 1e4

reforest_ha = ha("reforestable_weighted")
treated_ha  = ha("treated_as_forest")

# --- Unit costs (USD per ha) ---
ANR_MIN, ANR_MAX = 12, 3880
PLT_MIN, PLT_MAX = 105, 25830
ANR_MID = 0.5 * (ANR_MIN + ANR_MAX)
PLT_MID = 0.5 * (PLT_MIN + PLT_MAX)

scopes = [
    ("Reforestable only",        reforest_ha),
    ("Reforestable + treated",   reforest_ha + treated_ha),
]

rows = []
for scope, ha_val in scopes:
    rows += [
        {"scope": scope, "restoration_type": "ANR", "hectares": ha_val,
         "total_cost_min_usd": ha_val * ANR_MIN,
         "total_cost_mid_usd": ha_val * ANR_MID,
         "total_cost_max_usd": ha_val * ANR_MAX},
        {"scope": scope, "restoration_type": "Planted", "hectares": ha_val,
         "total_cost_min_usd": ha_val * PLT_MIN,
         "total_cost_mid_usd": ha_val * PLT_MID,
         "total_cost_max_usd": ha_val * PLT_MAX},
    ]

costs_usd = pd.DataFrame(rows)

# Pretty view in USD millions
disp = costs_usd.assign(
    **{f"{c} [USD mn]": costs_usd[c] / 1e6 for c in
       ("total_cost_min_usd","total_cost_mid_usd","total_cost_max_usd")}
).drop(columns=[c for c in costs_usd.columns if c.endswith("_usd")])

print(disp.to_string(index=False, formatters={"hectares":"{:,.1f}".format}))

# Optional save
fig_dir = base_path / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
costs_usd.to_csv(fig_dir / "restoration_cost_totals_raw_usd.csv", index=False)
disp.to_csv(fig_dir / "restoration_cost_totals_usd_mn.csv", index=False)
print("Saved:", fig_dir / "restoration_cost_totals_usd_mn.csv")

In [ ]:


# ---- Prep: rename for convenience (uses your 'disp' table in USD mn) ----
plot = disp.rename(columns={
    "total_cost_min [USD mn]": "min",
    "total_cost_mid [USD mn]": "mid",
    "total_cost_max [USD mn]": "max",
}).copy()

print(list(disp.columns))

# Order rows
type_order  = ["ANR", "Planted"]
scope_order = ["Reforestable only", "Reforestable + treated"]
plot["restoration_type"] = pd.Categorical(plot["restoration_type"], type_order)
plot["scope"] = pd.Categorical(plot["scope"], scope_order)
plot = plot.sort_values(["restoration_type", "scope"]).reset_index(drop=True)

# Custom y-axis labels
type_map  = {"ANR": "ANR", "Planted": "Plant"}
scope_map = {
    "Reforestable only": "reforest",
    "Reforestable + treated": "reforest & convert",
}
labels = [
    f"{type_map[str(t)]}: {scope_map[str(s)]}"
    for t, s in zip(plot["restoration_type"].astype(str), plot["scope"].astype(str))
]

# Colors by scope
scope_colors = {
    "Reforestable only": "#1f77b4",        # blue
    "Reforestable + treated": "#2ca02c",   # green
}

# ---- Plot: min–max interval + mid point (endpoint labels) ----
y = np.arange(len(plot))[::-1]  # top → bottom
fig, ax = plt.subplots(figsize=(8.6, 4.8))

dx = 0.01 * (plot["max"].max() - plot["min"].min())  # small outward offset

for i, r in plot.iterrows():
    c = scope_colors[str(r["scope"])]
    ax.hlines(y=y[i], xmin=r["min"], xmax=r["max"], color=c, linewidth=3, zorder=2)
    ax.scatter(r["mid"], y[i], s=48, color="white", edgecolor=c, linewidth=1.4, zorder=3)
    ax.text(r["min"] - dx, y[i], f"{r['min']:,.0f}", ha="right", va="center",
            fontsize=9, color=c, clip_on=False)
    ax.text(r["max"] + dx, y[i], f"{r['max']:,.0f}", ha="left", va="center",
            fontsize=9, color=c, clip_on=False)

ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel("Total establishment cost (USD mn)")
ax.grid(axis="x", linestyle=":", color="0.88")
ax.set_axisbelow(True)
ax.margins(x=0.06, y=0.10)

# Round-number ticks, numbers only
ax.xaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:,.0f}"))

# Legend
handles = [Line2D([0],[0], color=scope_colors[s], lw=4) for s in scope_order]
ax.legend(handles, scope_order, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
fig.subplots_adjust(right=0.8)
fig.tight_layout()

# Save (optional)
try:
    out_dir
except NameError:
    base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
    out_dir = base_path / "figures"
out_dir.mkdir(parents=True, exist_ok=True)
fname = out_dir / "restoration_costs_interval_min_max_labels_usd_mn"
fig.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(fname.with_suffix(".pdf"),              bbox_inches="tight")
print("Saved:", fname.with_suffix(".png"), "and", fname.with_suffix(".pdf"))

